# PhishExtension + PhishIntention (Logo detection) V2

## Settings

In [ ]:
websocket_HOST = "localhost"
websocket_PORT = 9997
classification_expiration_sec = 1 * 24 * 60 * 60  # Default 1 day
# classification_expiration_sec = 3  # In this case, 3 seconds

## Code

In [ ]:
import asyncio
import base64
import json
import os
import random
import shutil
import string
import time
from io import BytesIO

import numpy
import websockets
from PIL import Image

## Set up PhishIntention and folders
from phishintention_extension import PhishIntentionWrapper

print("Loading PI")
phishintention_cls = PhishIntentionWrapper()
print("Loaded PI")
shutil.rmtree("img_temp", ignore_errors=True)
os.makedirs("img_temp")

#### Image helper functions

In [ ]:
def base64_to_image(base64_string):
    # Remove the data URI prefix if present, although it should have already been removed below
    if "data:image" in base64_string:
        base64_string = base64_string.split(",")[1]

    # Decode the Base64 string into bytes
    image_bytes = base64.b64decode(base64_string)
    return image_bytes


def create_image_from_bytes(image_bytes):
    # Create a BytesIO object to handle the image data
    image_stream = BytesIO(image_bytes)

    # Open the image using PIL
    image = Image.open(image_stream)
    return image

#### Classification functions

In [ ]:
active_requests = {
    "example_id": {
        "hostname": "example2.com",
        "parameters": {"screenAmount": 5, "screenTiming": 1000, "screenTollerance": 10},
        "screen_results": [],
    }
}

classified_websites = {
    "example.com": {
        "status": "safe",
        "status_expire": int(time.time() + classification_expiration_sec) * 1000,
    }
}


def website_status_unknown(parsed_message):
    # Generate a 6 hex char string to use as identifier for the subsequent requests
    request_id = os.urandom(3).hex()
    hostname = parsed_message["hostname"]
    request_data = {
        "hostname": hostname,
        "parameters": parsed_message["parameters"],
        "screen_results": [],
    }
    active_requests[request_id] = request_data
    status_unknown_response = {
        "type": "status_unknown",
        "hostname": parsed_message["hostname"],
        "request_id": request_id,
    }
    return status_unknown_response


def check_website_status(parsed_message):
    check_status = get_status_if_available(parsed_message)
    if check_status:
        return check_status
    else:
        return website_status_unknown(parsed_message)


def get_status_if_available(parsed_message):
    hostname = parsed_message["hostname"]
    current_timestamp = int(time.time()) * 1000
    # If we have already recently classified the website
    if (hostname in classified_websites) and (
        current_timestamp < classified_websites[hostname]["status_expire"]
    ):
        status_known_response = {
            "type": "status_known",
            "hostname": hostname,
            "status": classified_websites[hostname]["status"],  # "safe" or "warn"
            "status_expire": classified_websites[hostname]["status_expire"],
        }
        print(
            hostname + " is KNOWN. Sending status: " + status_known_response["status"]
        )
        return status_known_response
    else:
        print(hostname + " is NOT known yet. Sending status_unknown message.")
        return None


def delete_request(request_id):
    return active_requests.pop(request_id, None)


def received_screen(parsed_message):
    request_id = parsed_message["request_id"]
    print(request_id)
    base64_screen = parsed_message["img_data"]
    screen_counter = parsed_message["screen_counter"]
    base64_screen = base64_screen.replace("data:image/png;base64,", "")

    image_bytes = base64_to_image(base64_screen)
    img = create_image_from_bytes(image_bytes)
    img_name = "img_temp/" + str(request_id) + "-" + str(screen_counter) + ".png"
    img.save(img_name)
    display(img)
    img.close()

    screen_results = analyze_screen(
        str(request_id) + "-" + str(screen_counter), img_name
    )
    # Check if the request is still active or if we received a screenshot for a deleted request
    if request_id not in active_requests:
        return "Error"  # We received a screen for a deleted request

    active_requests[request_id]["screen_results"].append(screen_results)
    if int(screen_counter) >= int(
        active_requests[request_id]["parameters"]["screenAmount"]
    ):
        assign_classification(request_id)

    return "OK"


def assign_classification(request_id):
    request = active_requests[request_id]
    is_safe = True

    # FIRST TEST: Different logo amounts
    first_logo_count = request["screen_results"][0]["logo_count"]
    for screen in request["screen_results"]:
        if screen["logo_count"] != first_logo_count:
            # Different amount of logos detected
            print("##DIFFERENT NUMBER OF LOGOS##")
            is_safe = False
            break

    # SECOND TEST: Check logo coordinates
    if is_safe and int(first_logo_count) > 0:
        tollerance = int(request["parameters"]["screenTollerance"])
        first_logo_coords = request["screen_results"][0]["logo_coords"]
        first_logo_coords = first_logo_coords.tolist()
        print("####BEGIN TESTING COORDINATES####")
        for screen in request["screen_results"]:
            if is_safe:
                logo_count = int(screen["logo_count"])
                logo_coords = screen["logo_coords"].tolist()
                print(logo_coords)
                # Try to match all logos, even if in different orders
                for i in range(0, first_logo_count):
                    print(first_logo_count)
                    print("DENTRO FOR i")
                    for k in range(
                        0, len(logo_coords)
                    ):  # Length changes during iterations
                        print("DENTRO FOR k")
                        # if k >= len(logo_coords):
                        #    break  # We have removed some elements
                        print("########## Logo coords: ")
                        print(first_logo_coords[i])
                        print(logo_coords[k])

                        matching = True
                        for point_a, point_b in zip(
                            first_logo_coords[i], logo_coords[k]
                        ):
                            print(point_a - point_b)
                            if abs(point_a - point_b) > tollerance:
                                print("OUTSIDE TOLLERANCE")
                                print(abs(point_a - point_b))
                                matching = False
                                break
                        print(matching)
                        print("##########")
                        if matching:
                            # logo_coords = numpy.delete(logo_coords, k)
                            print("MATCHED, REMOVING COORDINATES")
                            print(len(logo_coords))
                            logo_coords.pop(k)
                            print(len(logo_coords))
                            # k = k - 1
                            break
                    print(len(logo_coords))
                if len(logo_coords) != 0:
                    print("##LOGO MISMATCH##")
                    is_safe = False
                    break
            else:
                break

    hostname = request["hostname"]
    classified_websites[hostname] = {
        "status": "safe" if is_safe else "warn",
        "status_expire": int(time.time() + classification_expiration_sec) * 1000,
    }
    delete_request(request_id)


# Calls a modified version of PhishIntension to get logo data.
def analyze_screen(url, screenshot_path):
    (
        phish_category,
        pred_target,
        matched_domain,
        plotvis,
        siamese_conf,
        runtime_breakdown,
        pred_boxes,
        pred_classes,
        logo_pred_boxes,
    ) = phishintention_cls.test_orig_phishintention(url, screenshot_path)
    screen_results = {
        "logo_count": 0 if logo_pred_boxes is None else len(logo_pred_boxes),
        "logo_coords": logo_pred_boxes,
    }
    return screen_results

#### Websocket handling

In [ ]:
async def new_handler(websocket):
    async for message in websocket:
        parsed_message = json.loads(message)

        # Used by the extension to request the status of the current hostname.
        # If the status is available, a type="status_known" is sent with the classification info.
        # Otherwise, a type="status_unknown" is sent with a generated id.
        if parsed_message["type"] == "scan":
            status_message = check_website_status(parsed_message)
            print(status_message)
            await websocket.send(json.dumps(status_message))

        # Used when the extension has sent all screens and start querying the server to get the classification.
        elif parsed_message["type"] == "get_result":
            result = get_status_if_available(parsed_message)
            if result:
                await websocket.send(json.dumps(result))

        # Used by the extension to send the screen and the associated data
        elif parsed_message["type"] == "send_screen":
            received_screen(parsed_message)

        # Used by the extension when the process gets interrupted due to a tab change
        elif parsed_message["type"] == "abort_scan":
            delete_request(parsed_message["request_id"])
            # delete_request(parsed_message)

        # Used by the extension to keep the websocket connection open if idling
        elif parsed_message["type"] == "ping":
            pass

        else:
            error_response = {"type": "error", "info": "Invalid message type"}
            await websocket.send(json.dumps(error_response))


async def main():
    async with websockets.serve(
        new_handler, websocket_HOST, websocket_PORT, max_size=2 ** 28
    ):
        await asyncio.Future()  # Run forever


# The first function call below does not work when executed within Jupyter.

# asyncio.run(main())
await main()